In [ ]:
from hashlib import shake_256
from pathlib import Path
from typing import Sequence
import random
import time

import chipwhisperer as cw


Q = 127
N = 16

PRIVATE_KEY_SEED_LENGTH_BYTES = 32

FQ_ELEM_BYTES = 1
POSITION_T_BYTES = 2

COEFFICIENT_BYTES = (
    N * FQ_ELEM_BYTES
)

PERMUTATION_BYTES = (
    N * POSITION_T_BYTES
)

INPUT_BYTES = (
    PRIVATE_KEY_SEED_LENGTH_BYTES
)

OUTPUT_BYTES = (
    COEFFICIENT_BYTES
    + PERMUTATION_BYTES
)

SUBROUTINE_MONOMIAL_SAMPLE_PRIKEY = 0x03

BAUDRATE = 230400

FIRMWARE_DIRECTORY = Path.cwd()

HEX_BINARY = (
    FIRMWARE_DIRECTORY
    / "less_monomial_sample_prikey-PINATA.hex"
)

print("Seed bytes       :", INPUT_BYTES)
print("Coefficient bytes:", COEFFICIENT_BYTES)
print("Permutation bytes:", PERMUTATION_BYTES)
print("Output bytes     :", OUTPUT_BYTES)
print("HEX file         :", HEX_BINARY)

In [ ]:
class ShakeReader:
    """Sequential reader over Python's SHAKE256 output stream."""

    def __init__(
        self,
        seed: bytes,
    ) -> None:
        self._shake = shake_256(seed)
        self._offset = 0

    def read(
        self,
        length: int,
    ) -> bytes:
        end = self._offset + length

        data = self._shake.digest(end)[
            self._offset:end
        ]

        self._offset = end

        return data


def serialize_private_key_seed(
    seed: Sequence[int] | bytes | bytearray,
) -> bytes:
    try:
        seed_bytes = bytes(seed)
    except (TypeError, ValueError) as exc:
        raise TypeError(
            "Seed must be bytes or a sequence of byte values"
        ) from exc

    if len(seed_bytes) != PRIVATE_KEY_SEED_LENGTH_BYTES:
        raise ValueError(
            f"Expected {PRIVATE_KEY_SEED_LENGTH_BYTES} seed bytes, "
            f"got {len(seed_bytes)}"
        )

    return seed_bytes


def monomial_sample_prikey_python(
    seed: Sequence[int] | bytes | bytearray,
) -> tuple[list[int], list[int]]:
    """Independent reproduction of monomial_sample_prikey."""

    seed_bytes = serialize_private_key_seed(
        seed
    )

    reader = ShakeReader(
        seed_bytes
    )

    coefficients: list[int] = []

    while len(coefficients) < N:
        word = int.from_bytes(
            reader.read(8),
            byteorder="little",
        )

        for _ in range(64 // 7):
            candidate = word & 0x7F

            if candidate <= Q - 2:
                coefficients.append(
                    candidate + 1
                )

                if len(coefficients) == N:
                    break

            word >>= 7

    permutation = list(
        range(N)
    )

    position_bits = (
        N - 1
    ).bit_length()

    position_mask = (
        1 << position_bits
    ) - 1

    random_word = int.from_bytes(
        reader.read(8),
        byteorder="little",
    )

    consumed_candidates = 0

    for index in range(N):
        while True:
            if consumed_candidates == (
                64 // position_bits
            ) - 1:
                random_word = int.from_bytes(
                    reader.read(8),
                    byteorder="little",
                )

                consumed_candidates = 0

            destination = (
                random_word
                & position_mask
            )

            random_word >>= position_bits

            consumed_candidates += 1

            if destination < N:
                break

        permutation[index], permutation[destination] = (
            permutation[destination],
            permutation[index],
        )

    return (
        coefficients,
        permutation,
    )


def deserialize_monomial_result(
    payload: bytes | bytearray,
) -> tuple[list[int], list[int]]:
    payload_bytes = bytes(
        payload
    )

    if len(payload_bytes) != OUTPUT_BYTES:
        raise ValueError(
            f"Expected {OUTPUT_BYTES} result bytes, "
            f"got {len(payload_bytes)}"
        )

    coefficients = list(
        payload_bytes[:COEFFICIENT_BYTES]
    )

    permutation_data = payload_bytes[
        COEFFICIENT_BYTES:
    ]

    permutation = [
        int.from_bytes(
            permutation_data[
                POSITION_T_BYTES * index:
                POSITION_T_BYTES * (index + 1)
            ],
            byteorder="little",
            signed=False,
        )
        for index in range(N)
    ]

    return (
        coefficients,
        permutation,
    )

In [ ]:
def connect_and_flash():
    if not HEX_BINARY.is_file():
        raise FileNotFoundError(
            f"Firmware image not found: {HEX_BINARY}"
        )

    scope = cw.scope()
    scope.default_setup()

    scope.io.clkout = 15e6

    print(
        "Clock output:",
        scope.io.clkout,
    )

    time.sleep(0.5)

    cw.program_target(
        scope,
        cw.programmers.STM32FProgrammer,
        str(HEX_BINARY),
        baud=BAUDRATE,
    )

    target = cw.target(
        scope,
        cw.targets.SimpleSerial2,
    )

    target.baud = BAUDRATE
    target.flush()

    return (
        scope,
        target,
    )


def upload_private_key_seed(
    target,
    seed: Sequence[int] | bytes | bytearray,
    verify_readback: bool = True,
) -> None:
    payload = serialize_private_key_seed(
        seed
    )

    subcommand = (
        0x80
        if verify_readback
        else 0x00
    )

    target.send_cmd(
        "s",
        subcommand,
        bytearray(payload),
    )

    if verify_readback:
        readback = target.simpleserial_read(
            "r",
            INPUT_BYTES,
            timeout=1000,
            ack=True,
        )

        if readback is None:
            raise TimeoutError(
                "No private-key seed readback received"
            )

        if bytes(readback) != payload:
            raise RuntimeError(
                "Private-key seed readback mismatch"
            )

    else:
        acknowledgement = (
            target.simpleserial_wait_ack(
                timeout=1000
            )
        )

        if acknowledgement is None:
            raise TimeoutError(
                "No acknowledgement for private-key seed"
            )


def monomial_sample_prikey_board(
    target,
    seed: Sequence[int] | bytes | bytearray,
) -> tuple[list[int], list[int]]:
    upload_private_key_seed(
        target,
        seed,
        verify_readback=True,
    )

    target.send_cmd(
        "b",
        SUBROUTINE_MONOMIAL_SAMPLE_PRIKEY,
        bytearray(),
    )

    response = target.simpleserial_read(
        "r",
        OUTPUT_BYTES,
        timeout=2000,
        ack=True,
    )

    if response is None:
        raise TimeoutError(
            "No result received from monomial_sample_prikey"
        )

    if len(response) != OUTPUT_BYTES:
        raise RuntimeError(
            f"Expected {OUTPUT_BYTES} result bytes, "
            f"received {len(response)}"
        )

    return deserialize_monomial_result(
        response
    )

In [ ]:
scope = None
target = None

try:
    scope, target = connect_and_flash()

    deterministic_seed = bytes(
        range(PRIVATE_KEY_SEED_LENGTH_BYTES)
    )

    expected = monomial_sample_prikey_python(
        deterministic_seed
    )

    actual = monomial_sample_prikey_board(
        target,
        deterministic_seed,
    )

    print(
        "Expected coefficients:",
        expected[0],
    )

    print(
        "Board coefficients   :",
        actual[0],
    )

    print(
        "Expected permutation:",
        expected[1],
    )

    print(
        "Board permutation   :",
        actual[1],
    )

    if actual != expected:
        raise AssertionError(
            "Pinata result does not match the Python result"
        )

    number_of_random_tests = 100

    for test_index in range(
        number_of_random_tests
    ):
        seed = bytes(
            random.randrange(256)
            for _ in range(
                PRIVATE_KEY_SEED_LENGTH_BYTES
            )
        )

        expected = monomial_sample_prikey_python(
            seed
        )

        actual = monomial_sample_prikey_board(
            target,
            seed,
        )

        if actual != expected:
            raise AssertionError(
                f"Mismatch in random test {test_index}"
            )

    print(
        f"OK: deterministic test and "
        f"{number_of_random_tests} random tests passed."
    )

finally:
    if target is not None:
        target.dis()

    if scope is not None:
        scope.dis()